Part A

Classic Word2Vec stores one embedding per token, regardless of meaning.
So:
cheap = affordable
cheap = low-quality
both get merged into the same vector.

Cosine Similarity Results

Since words like affordable and flimsy were not present in the dataset vocabulary, I used nearest semantic anchors from the corpus:

Affordable sense → price
Low-quality sense → poor

| Comparison     | Cosine Similarity |
| -------------- | ----------------- |
| cheap vs price | **0.674**         |
| cheap vs poor  | **0.631**         |


The vector for cheap is simultaneously close to:

price/value meaning
poor-quality meaning

This confirms that Word2Vec collapses both senses into one embedding.


Q.b

We classify:

Sense 1: Affordable

Examples:

cheap price
cheap and worth it
very cheap product
Sense 2: Low-quality

Examples:

cheap plastic
feels cheap
cheap material

In [2]:
import pandas as pd
import numpy as np
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec
from numpy.linalg import norm


df = pd.read_csv("/Users/omshinde/Desktop/Python/assignment/Assignment 38/shopsense_reviews.csv")
df["review_text"] = df["review_text"].fillna("").astype(str)
sentences = [simple_preprocess(t) for t in df["review_text"] if t.strip() != ""]

# Train model
model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,
    epochs=15
)

wv = model.wv

def cosine(a, b):
    return np.dot(a,b)/(norm(a)*norm(b))

# Build anchors
affordable_words = ["price", "value", "worth"]
low_words = ["poor", "material", "quality"]

aff_anchor = np.mean([wv[w] for w in affordable_words if w in wv], axis=0)
low_anchor = np.mean([wv[w] for w in low_words if w in wv], axis=0)

def sentence_vector(sentence):
    words = simple_preprocess(sentence)
    words = [w for w in words if w != "cheap" and w in wv]
    return np.mean([wv[w] for w in words], axis=0)

def classify(sentence):
    vec = sentence_vector(sentence)

    s1 = cosine(vec, aff_anchor)
    s2 = cosine(vec, low_anchor)

    if s1 > s2:
        return "AFFORDABLE", s1, s2
    else:
        return "LOW_QUALITY", s1, s2

# Test
tests = [
    "This phone is cheap and worth every rupee",
    "The bag feels cheap and poor quality",
    "Cheap price but good value",
    "Cheap material and bad finishing"
]

for t in tests:
    print(t, "->", classify(t))

This phone is cheap and worth every rupee -> ('AFFORDABLE', np.float32(0.74362093), np.float32(0.5183785))
The bag feels cheap and poor quality -> ('LOW_QUALITY', np.float32(0.5753709), np.float32(0.8230677))
Cheap price but good value -> ('AFFORDABLE', np.float32(0.79221886), np.float32(0.5928203))
Cheap material and bad finishing -> ('LOW_QUALITY', np.float32(0.7134737), np.float32(0.8128064))


`window_size = 2` uses nearby words only, so it captures more **syntactic relationships** like phrase patterns and grammar (example: cheap material, poor quality).
`window_size = 10` uses a wider context, so it captures more **semantic relationships** like overall meaning and topic similarity (example: cheap, price, value).
Small windows are better for local structure, while large windows are better for broader meaning.


Part 2

In [5]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sentence_transformers import SentenceTransformer

review_a = "incredible camera but terrible battery life"
review_b = "Battery drains fast, although photos are stunning"

docs = [review_a, review_b]


bow = CountVectorizer()
X_bow = bow.fit_transform(docs)
bow_sim = cosine_similarity(X_bow[0], X_bow[1])[0][0]

print("BOW Vocabulary:", bow.get_feature_names_out())
print("BOW Cosine Similarity:", round(bow_sim, 4))


tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(docs)
tfidf_sim = cosine_similarity(X_tfidf[0], X_tfidf[1])[0][0]

print("\nTF-IDF Cosine Similarity:", round(tfidf_sim, 4))


tokenized = [simple_preprocess(doc) for doc in docs]


w2v = Word2Vec(
    sentences=tokenized,
    vector_size=100,
    window=5,
    min_count=1,
    workers=1,
    sg=1,
    epochs=200
)

def avg_vector(tokens, model):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    return np.mean(vecs, axis=0)

vec_a = avg_vector(tokenized[0], w2v)
vec_b = avg_vector(tokenized[1], w2v)

w2v_sim = cosine_similarity([vec_a], [vec_b])[0][0]

print("Word2Vec Avg Cosine Similarity:", round(w2v_sim, 4))



model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(docs)
sbert_sim = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]

print("Sentence-BERT Cosine Similarity:", round(sbert_sim, 4))


print("\n--- Final Similarity Scores ---")
print("BOW       :", round(bow_sim, 4))
print("TF-IDF    :", round(tfidf_sim, 4))
print("Word2Vec  :", round(w2v_sim, 4))
print("SBERT     :", round(sbert_sim, 4))

BOW Vocabulary: ['although' 'are' 'battery' 'but' 'camera' 'drains' 'fast' 'incredible'
 'life' 'photos' 'stunning' 'terrible']
BOW Cosine Similarity: 0.1543

TF-IDF Cosine Similarity: 0.0846
Word2Vec Avg Cosine Similarity: 0.1411


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence-BERT Cosine Similarity: 0.6336

--- Final Similarity Scores ---
BOW       : 0.1543
TF-IDF    : 0.0846
Word2Vec  : 0.1411
SBERT     : 0.6336


### (a) Which correctly identifies similarity?

**Sentence-BERT** identifies the similarity best because it understands overall meaning, context, and synonyms like *camera ↔ photos* and *battery life ↔ drains fast*. **Word2Vec averaging** performs better than BOW and TF-IDF because it captures word relationships, but it loses sentence structure. **BOW** and **TF-IDF** rely mostly on exact word overlap, so they may give low similarity even when both reviews mean the same thing.

### (b) Why does BOW fail?

Bag of Words compares only shared words. Here, the only common word is **battery**, while words like **camera** and **photos** or **terrible battery life** and **drains fast** are treated as completely different. Since it ignores meaning and context, it wrongly assumes the reviews are less similar.

### (c) Semantic Gap Explanation

The **semantic gap** means two texts can express the same idea using different words. **BOW** cannot bridge this gap because it uses raw word counts. **TF-IDF** slightly improves weighting important words, **Word2Vec** captures semantic similarity between related words, and **Sentence-BERT** closes the gap best by understanding full sentence meaning and context.